# Оценка погрешности перемещения по IMU относительно GPS/POS

Цель ноутбука - быстро войти в данные, собрать воспроизводимый baseline и подготовить основу для проверки гипотез: можно ли по инерциальным и вспомогательным датчикам оценивать смещение БПЛА без использования GPS/GNSS на входе.

Ключевое правило эксперимента: `GPS`/`POS` используются только как эталонная траектория и target для проверки ошибки. Во входные признаки модели их не добавляем.

Минимальные зависимости для этого ноутбука: `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `jupyter`. Полный список проекта лежит в `requirements.txt`, но для стартового анализа `tensorflow` не нужен.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cwd = Path.cwd().resolve()
if (cwd / 'derived').exists():
    ROOT = cwd
elif (cwd.parent / 'derived').exists():
    ROOT = cwd.parent
else:
    ROOT = Path('..').resolve()
DATAFLASH = ROOT / 'derived' / 'dataflash'
REPORTS = ROOT / 'reports'

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

DATAFLASH

## 1. Инвентаризация источников

В репозитории уже есть подготовленный экспорт DataFlash. Для стартовой проверки берем его, потому что там есть синхронизируемые ряды `IMU`, `ATT`, `BARO`, `BAT`, `MOTB`, `RCOU` и эталон `POS/GPS`.

Практическая постановка для первой гипотезы:

- вход: история сенсоров за последние `lookback_s` секунд;
- target: смещение `dx_east`, `dy_north`, `dz_up` через `horizon_s` секунд по `POS`;
- split: хронологический, чтобы будущие фрагменты не попадали в обучение.

In [ ]:
flight_index_path = ROOT / 'derived' / 'datasets' / 'flight_index.csv'
if flight_index_path.exists():
    flight_index = pd.read_csv(flight_index_path)
    display(flight_index[['flight_id', 'source_format', 'duration_s', 'gps_points', 'sensor_rows', 'gps_distance_2d_m']])
else:
    print('flight_index.csv не найден')

for name in ['IMU', 'ATT', 'BARO', 'BAT', 'MOTB', 'RCOU', 'POS', 'GPS']:
    path = DATAFLASH / f'{name}.csv'
    if path.exists():
        rows = sum(1 for _ in path.open('r', encoding='utf-8')) - 1
        print(f'{name:5s}: {rows:6d} rows -> {path.relative_to(ROOT)}')

## 2. Загрузка и перевод эталонной траектории в локальные метры

`POS.Lat/Lng/Alt` переводим в локальную ENU-систему: east, north, up относительно первой валидной точки. Это удобнее для ошибок в метрах и для target-смещений.

In [ ]:
def read_dataflash_csv(name: str) -> pd.DataFrame:
    df = pd.read_csv(DATAFLASH / f'{name}.csv')
    if 'TimeUS' in df.columns:
        df['time_s'] = df['TimeUS'] / 1_000_000.0
        df['time_rel_s'] = df['time_s'] - df['time_s'].iloc[0]
    return df

def latlon_to_enu_m(lat, lon, alt):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    alt = np.asarray(alt, dtype=float)
    lat0, lon0, alt0 = lat[0], lon[0], alt[0]
    earth_radius_m = 6_371_000.0
    north = np.deg2rad(lat - lat0) * earth_radius_m
    east = np.deg2rad(lon - lon0) * earth_radius_m * np.cos(np.deg2rad(lat0))
    up = alt - alt0
    return east, north, up

imu = read_dataflash_csv('IMU')
att = read_dataflash_csv('ATT')
baro = read_dataflash_csv('BARO')
pos = read_dataflash_csv('POS')
gps = read_dataflash_csv('GPS')

pos['east_m'], pos['north_m'], pos['up_m'] = latlon_to_enu_m(pos['Lat'], pos['Lng'], pos['Alt'])
gps['east_m'], gps['north_m'], gps['up_m'] = latlon_to_enu_m(gps['Lat'], gps['Lng'], gps['Alt'])

summary = pd.DataFrame({
    'rows': [len(imu), len(att), len(baro), len(pos), len(gps)],
    'duration_s': [df['time_s'].iloc[-1] - df['time_s'].iloc[0] for df in [imu, att, baro, pos, gps]],
}, index=['IMU', 'ATT', 'BARO', 'POS', 'GPS'])
summary['rate_hz'] = summary['rows'] / summary['duration_s']
display(summary.round(3))
display(pos[['time_rel_s', 'Lat', 'Lng', 'Alt', 'east_m', 'north_m', 'up_m']].head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(pos['east_m'], pos['north_m'], label='POS')
axes[0].plot(gps['east_m'], gps['north_m'], '.', markersize=2, alpha=0.45, label='GPS')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel('east, m')
axes[0].set_ylabel('north, m')
axes[0].legend()

axes[1].plot(pos['time_rel_s'], pos['up_m'], label='POS up')
axes[1].plot(gps['time_rel_s'], gps['up_m'], '.', markersize=2, alpha=0.45, label='GPS up')
axes[1].set_xlabel('time, s')
axes[1].set_ylabel('up, m')
axes[1].legend()
plt.show()

pos_distance = np.hypot(pos['east_m'].diff(), pos['north_m'].diff()).fillna(0).sum()
print(f'POS horizontal path length: {pos_distance:.1f} m')
print(f'POS duration: {pos.time_rel_s.iloc[-1]:.1f} s')

## 3. Быстрая диагностика IMU/ATT/BARO

Перед моделями проверяем базовые масштабы: нормы ускорения/гироскопа, углы ориентации и барометрическую высоту. Это помогает увидеть зависания, скачки, насыщение и участки активного движения.

In [ ]:
imu['acc_norm'] = np.sqrt(imu['AccX']**2 + imu['AccY']**2 + imu['AccZ']**2)
imu['gyro_norm'] = np.sqrt(imu['GyrX']**2 + imu['GyrY']**2 + imu['GyrZ']**2)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(imu['time_rel_s'], imu['acc_norm'])
axes[0].set_ylabel('|acc|, m/s^2')
axes[1].plot(imu['time_rel_s'], imu['gyro_norm'])
axes[1].set_ylabel('|gyro|, rad/s')
axes[2].plot(att['time_rel_s'], att[['Roll', 'Pitch', 'Yaw']])
axes[2].set_ylabel('angle, deg')
axes[2].set_xlabel('time, s')
axes[2].legend(['Roll', 'Pitch', 'Yaw'], ncol=3)
plt.show()

display(imu[['AccX', 'AccY', 'AccZ', 'acc_norm', 'GyrX', 'GyrY', 'GyrZ', 'gyro_norm']].describe().T.round(4))

## 4. Точность барометра относительно POS/GPS

Накладываем барометрическую высоту на эталон `POS/GPS`. Сравниваем два варианта:

- `BARO.AltAMSL` против абсолютной высоты `POS.Alt`/`GPS.Alt`;
- `BARO.Alt` против относительной высоты `POS up`.

Барометр обычно хорошо ловит форму изменения высоты, но имеет произвольное смещение. Поэтому ниже считаем offset по первым 30 секундам и отдельно смотрим остаточную ошибку и drift.

In [ ]:
def altitude_error_metrics(reference_m, estimate_m) -> dict:
    reference_m = np.asarray(reference_m, dtype=float)
    estimate_m = np.asarray(estimate_m, dtype=float)
    err = estimate_m - reference_m
    return {
        'bias_m': err.mean(),
        'mae_m': np.abs(err).mean(),
        'rmse_m': np.sqrt(np.mean(err**2)),
        'p95_abs_m': np.percentile(np.abs(err), 95),
        'min_err_m': err.min(),
        'max_err_m': err.max(),
        'drift_m': err[-1] - err[0],
    }

calibration_s = 30.0
BARO_OUT = ROOT / 'artifacts' / 'generated' / 'barometer_altitude'
BARO_OUT.mkdir(parents=True, exist_ok=True)

baro_for_cmp = baro[['time_s', 'time_rel_s', 'Alt', 'AltAMSL', 'Press', 'Temp', 'CRt']].rename(columns={
    'Alt': 'baro_rel_m',
    'AltAMSL': 'baro_amsl_m',
}).sort_values('time_s')
pos_for_cmp = pos[['time_s', 'east_m', 'north_m', 'Alt', 'up_m']].rename(columns={
    'Alt': 'pos_amsl_m',
    'up_m': 'pos_rel_m',
}).sort_values('time_s')
gps_for_cmp = gps[['time_s', 'Alt', 'up_m']].rename(columns={
    'Alt': 'gps_amsl_m',
    'up_m': 'gps_rel_m',
}).sort_values('time_s')

baro_cmp = pd.merge_asof(
    baro_for_cmp,
    pos_for_cmp,
    on='time_s',
    direction='nearest',
    tolerance=0.20,
)
baro_cmp = pd.merge_asof(
    baro_cmp,
    gps_for_cmp,
    on='time_s',
    direction='nearest',
    tolerance=0.35,
)
baro_cmp = baro_cmp.dropna(subset=['pos_amsl_m', 'pos_rel_m']).reset_index(drop=True)
baro_cmp['time_rel_s'] = baro_cmp['time_s'] - baro_cmp['time_s'].iloc[0]

cal_mask = baro_cmp['time_rel_s'] <= calibration_s
rel_offset_m = (baro_cmp.loc[cal_mask, 'pos_rel_m'] - baro_cmp.loc[cal_mask, 'baro_rel_m']).median()
amsl_offset_m = (baro_cmp.loc[cal_mask, 'pos_amsl_m'] - baro_cmp.loc[cal_mask, 'baro_amsl_m']).median()
baro_cmp['baro_rel_cal_m'] = baro_cmp['baro_rel_m'] + rel_offset_m
baro_cmp['baro_amsl_cal_m'] = baro_cmp['baro_amsl_m'] + amsl_offset_m
baro_cmp['baro_rel_err_m'] = baro_cmp['baro_rel_cal_m'] - baro_cmp['pos_rel_m']
baro_cmp['baro_amsl_err_m'] = baro_cmp['baro_amsl_cal_m'] - baro_cmp['pos_amsl_m']
baro_cmp['baro_rel_err_roll_m'] = baro_cmp['baro_rel_err_m'].rolling(50, min_periods=1).mean()

metrics_baro = pd.DataFrame({
    'BARO Alt + offset vs POS up': altitude_error_metrics(baro_cmp['pos_rel_m'], baro_cmp['baro_rel_cal_m']),
    'BARO AltAMSL + offset vs POS Alt': altitude_error_metrics(baro_cmp['pos_amsl_m'], baro_cmp['baro_amsl_cal_m']),
}).T
metrics_baro.to_csv(BARO_OUT / 'barometer_metrics.csv')
baro_cmp.to_csv(BARO_OUT / 'barometer_pos_overlay.csv', index=False)

print(f'Rows compared: {len(baro_cmp)}')
print(f'Relative altitude offset from first {calibration_s:.0f}s: {rel_offset_m:.3f} m')
print(f'AMSL altitude offset from first {calibration_s:.0f}s: {amsl_offset_m:.3f} m')
print(f'Saved metrics: {(BARO_OUT / "barometer_metrics.csv").relative_to(ROOT)}')
print(f'Saved overlay data: {(BARO_OUT / "barometer_pos_overlay.csv").relative_to(ROOT)}')
display(metrics_baro.round(3))
display(baro_cmp[['time_rel_s', 'baro_rel_cal_m', 'pos_rel_m', 'baro_rel_err_m', 'baro_amsl_cal_m', 'pos_amsl_m', 'baro_amsl_err_m']].head())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True, constrained_layout=True)

axes[0].plot(pos['time_rel_s'], pos['up_m'], label='POS relative up', linewidth=1.8)
axes[0].plot(gps['time_rel_s'], gps['up_m'], '.', markersize=2, alpha=0.35, label='GPS relative up')
axes[0].plot(baro_cmp['time_rel_s'], baro_cmp['baro_rel_cal_m'], label='BARO Alt + offset', linewidth=1.0)
axes[0].set_title('Full-flight relative altitude overlay')
axes[0].set_ylabel('relative altitude, m')
axes[0].legend(ncol=3)

axes[1].plot(pos['time_rel_s'], pos['Alt'], label='POS Alt', linewidth=1.8)
axes[1].plot(gps['time_rel_s'], gps['Alt'], '.', markersize=2, alpha=0.35, label='GPS Alt')
axes[1].plot(baro_cmp['time_rel_s'], baro_cmp['baro_amsl_cal_m'], label='BARO AltAMSL + offset', linewidth=1.0)
axes[1].set_ylabel('absolute altitude, m')
axes[1].legend(ncol=3)

axes[2].plot(baro_cmp['time_rel_s'], baro_cmp['baro_rel_err_m'], alpha=0.35, label='instant error')
axes[2].plot(baro_cmp['time_rel_s'], baro_cmp['baro_rel_err_roll_m'], label='rolling mean error, 5s', linewidth=2.0)
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_ylabel('BARO - POS, m')
axes[2].set_xlabel('time, s')
axes[2].legend()

caption = (
    f"Offset by first {calibration_s:.0f}s: {rel_offset_m:+.3f} m | "
    f"MAE {metrics_baro.loc['BARO Alt + offset vs POS up', 'mae_m']:.3f} m | "
    f"RMSE {metrics_baro.loc['BARO Alt + offset vs POS up', 'rmse_m']:.3f} m | "
    f"P95 {metrics_baro.loc['BARO Alt + offset vs POS up', 'p95_abs_m']:.3f} m | "
    f"drift {metrics_baro.loc['BARO Alt + offset vs POS up', 'drift_m']:+.3f} m"
)
fig.text(0.5, 0.005, caption, ha='center', fontsize=11)
height_png = BARO_OUT / 'height_full_overlay.png'
fig.savefig(height_png, dpi=160)
print(f'Saved image: {height_png.relative_to(ROOT)}')
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(baro_cmp['pos_rel_m'], baro_cmp['baro_rel_cal_m'], s=6, alpha=0.35)
lims = [min(baro_cmp['pos_rel_m'].min(), baro_cmp['baro_rel_cal_m'].min()), max(baro_cmp['pos_rel_m'].max(), baro_cmp['baro_rel_cal_m'].max())]
ax.plot(lims, lims, color='black', linewidth=1)
ax.set_xlabel('POS relative up, m')
ax.set_ylabel('BARO Alt + offset, m')
ax.set_title('Relative altitude agreement')
ax.set_aspect('equal', adjustable='box')
scatter_png = BARO_OUT / 'height_scatter_pos_vs_baro.png'
fig.savefig(scatter_png, dpi=160)
print(f'Saved image: {scatter_png.relative_to(ROOT)}')
plt.show()

def colored_route(ax, x, y, values, cmap, norm, linewidth=2.2):
    points = np.column_stack([x, y]).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    line_collection = LineCollection(segments, cmap=cmap, norm=norm, linewidth=linewidth)
    line_collection.set_array(np.asarray(values[:-1]))
    ax.add_collection(line_collection)
    ax.autoscale()
    return line_collection

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
vmin = min(pos['up_m'].min(), baro_cmp['baro_rel_cal_m'].min())
vmax = max(pos['up_m'].max(), baro_cmp['baro_rel_cal_m'].max())
alt_norm = plt.Normalize(vmin=vmin, vmax=vmax)

lc0 = colored_route(axes[0], pos['east_m'].to_numpy(), pos['north_m'].to_numpy(), pos['up_m'].to_numpy(), 'viridis', alt_norm)
axes[0].set_title('POS route colored by height')
axes[0].set_xlabel('east, m')
axes[0].set_ylabel('north, m')
axes[0].set_aspect('equal', adjustable='box')
fig.colorbar(lc0, ax=axes[0], label='relative height, m')

lc1 = colored_route(axes[1], baro_cmp['east_m'].to_numpy(), baro_cmp['north_m'].to_numpy(), baro_cmp['baro_rel_cal_m'].to_numpy(), 'viridis', alt_norm)
axes[1].set_title('Same route colored by BARO height')
axes[1].set_xlabel('east, m')
axes[1].set_ylabel('north, m')
axes[1].set_aspect('equal', adjustable='box')
fig.colorbar(lc1, ax=axes[1], label='relative height, m')

err_abs = max(abs(baro_cmp['baro_rel_err_m'].min()), abs(baro_cmp['baro_rel_err_m'].max()), 1.0)
err_norm = plt.Normalize(vmin=-err_abs, vmax=err_abs)
lc2 = colored_route(axes[2], baro_cmp['east_m'].to_numpy(), baro_cmp['north_m'].to_numpy(), baro_cmp['baro_rel_err_m'].to_numpy(), 'coolwarm', err_norm)
axes[2].set_title('Route colored by BARO - POS error')
axes[2].set_xlabel('east, m')
axes[2].set_ylabel('north, m')
axes[2].set_aspect('equal', adjustable='box')
fig.colorbar(lc2, ax=axes[2], label='height error, m')

route_png = BARO_OUT / 'route_height_error_overlay.png'
fig.savefig(route_png, dpi=160)
print(f'Saved image: {route_png.relative_to(ROOT)}')
plt.show()

## 5. Сбор supervised-таблицы

Синхронизируем ряды по времени через `merge_asof`, берем признаки только из сенсоров, а target строим по `POS` как будущее смещение через фиксированный горизонт. Для первого прохода используем простые rolling-агрегаты за окно `lookback_s`.

In [ ]:
def add_future_pos_targets(base: pd.DataFrame, pos_ref: pd.DataFrame, horizon_s: float) -> pd.DataFrame:
    out = base.copy()
    pos_time = pos_ref['time_s'].to_numpy()
    for col in ['east_m', 'north_m', 'up_m']:
        now = np.interp(out['time_s'], pos_time, pos_ref[col])
        future = np.interp(out['time_s'] + horizon_s, pos_time, pos_ref[col])
        axis = {'east_m': 'dx_east_m', 'north_m': 'dy_north_m', 'up_m': 'dz_up_m'}[col]
        out[axis] = future - now
        out[f'{col}_ref'] = now
    return out

def build_feature_table(horizon_s=5.0, lookback_s=5.0, sample_hz=10):
    step_s = 1.0 / sample_hz
    t0 = max(imu['time_s'].min(), att['time_s'].min(), baro['time_s'].min(), pos['time_s'].min())
    t1 = min(imu['time_s'].max(), att['time_s'].max(), baro['time_s'].max(), pos['time_s'].max() - horizon_s)
    grid = pd.DataFrame({'time_s': np.arange(t0, t1, step_s)})

    imu_cols = ['time_s', 'AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ', 'acc_norm', 'gyro_norm', 'T']
    att_cols = ['time_s', 'Roll', 'Pitch', 'Yaw']
    baro_cols = ['time_s', 'Alt', 'Press', 'Temp', 'CRt']

    df = pd.merge_asof(grid, imu[imu_cols].sort_values('time_s'), on='time_s', direction='nearest', tolerance=0.08)
    df = pd.merge_asof(df, att[att_cols].sort_values('time_s'), on='time_s', direction='nearest', tolerance=0.15)
    df = pd.merge_asof(df, baro[baro_cols].sort_values('time_s'), on='time_s', direction='nearest', tolerance=0.15, suffixes=('', '_baro'))
    df = df.dropna().reset_index(drop=True)
    df['time_rel_s'] = df['time_s'] - df['time_s'].iloc[0]

    sensor_cols = ['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ', 'acc_norm', 'gyro_norm', 'T', 'Roll', 'Pitch', 'Yaw', 'Alt', 'Press', 'Temp', 'CRt']
    window = max(2, int(round(lookback_s * sample_hz)))
    feature_parts = [df[['time_s', 'time_rel_s'] + sensor_cols].copy()]
    rolled = df[sensor_cols].rolling(window=window, min_periods=window)
    for suffix, values in {
        'mean': rolled.mean(),
        'std': rolled.std(),
        'min': rolled.min(),
        'max': rolled.max(),
    }.items():
        values = values.add_suffix(f'_{suffix}')
        feature_parts.append(values)
    features = pd.concat(feature_parts, axis=1).dropna().reset_index(drop=True)
    features = add_future_pos_targets(features, pos, horizon_s=horizon_s)
    return features

horizon_s = 5.0
lookback_s = 5.0
features = build_feature_table(horizon_s=horizon_s, lookback_s=lookback_s, sample_hz=10)
target_cols = ['dx_east_m', 'dy_north_m', 'dz_up_m']
feature_cols = [c for c in features.columns if c not in ['time_s', 'time_rel_s', 'east_m_ref', 'north_m_ref', 'up_m_ref'] + target_cols]

print(features.shape)
print(f'{len(feature_cols)} feature columns')
display(features[['time_rel_s'] + target_cols].describe().T.round(3))

## 6. Первый baseline: Ridge против нулевого смещения

Это не финальная модель, а sanity-check. Если модель не обгоняет `zero displacement`, признаки/target/split нужно пересматривать до усложнения архитектуры.

In [ ]:
def displacement_metrics(y_true, y_pred) -> dict:
    err = y_pred - y_true
    err_3d = np.linalg.norm(err, axis=1)
    return {
        'MAE east': mean_absolute_error(y_true[:, 0], y_pred[:, 0]),
        'MAE north': mean_absolute_error(y_true[:, 1], y_pred[:, 1]),
        'MAE up': mean_absolute_error(y_true[:, 2], y_pred[:, 2]),
        'MAE 3D': err_3d.mean(),
        'RMSE 3D': np.sqrt(np.mean(err_3d**2)),
        'P95 3D': np.percentile(err_3d, 95),
    }

split_time = features['time_rel_s'].quantile(0.80)
train_mask = features['time_rel_s'] <= split_time
test_mask = ~train_mask

X_train = features.loc[train_mask, feature_cols].to_numpy()
X_test = features.loc[test_mask, feature_cols].to_numpy()
y_train = features.loc[train_mask, target_cols].to_numpy()
y_test = features.loc[test_mask, target_cols].to_numpy()

ridge = make_pipeline(StandardScaler(), Ridge(alpha=100.0))
ridge.fit(X_train, y_train)

pred_zero = np.zeros_like(y_test)
pred_mean = np.repeat(y_train.mean(axis=0, keepdims=True), len(y_test), axis=0)
pred_ridge = ridge.predict(X_test)

metrics = pd.DataFrame({
    'zero': displacement_metrics(y_test, pred_zero),
    'train_mean': displacement_metrics(y_test, pred_mean),
    'ridge': displacement_metrics(y_test, pred_ridge),
}).T
display(metrics.round(3))
print(f'train rows: {len(X_train)}, test rows: {len(X_test)}, split time: {split_time:.1f} s')

In [ ]:
test_view = features.loc[test_mask, ['time_rel_s'] + target_cols].copy()
for i, col in enumerate(target_cols):
    test_view[f'pred_{col}'] = pred_ridge[:, i]
    test_view[f'err_{col}'] = pred_ridge[:, i] - y_test[:, i]
test_view['err_3d_m'] = np.linalg.norm(pred_ridge - y_test, axis=1)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for col in target_cols:
    axes[0].plot(test_view['time_rel_s'], test_view[col], label=f'true {col}')
    axes[0].plot(test_view['time_rel_s'], test_view[f'pred_{col}'], '--', label=f'pred {col}')
axes[0].set_ylabel('5s displacement, m')
axes[0].legend(ncol=3, fontsize=8)

axes[1].plot(test_view['time_rel_s'], test_view['err_3d_m'])
axes[1].set_ylabel('3D error, m')
axes[1].set_xlabel('time, s')
plt.show()

display(test_view[['time_rel_s', 'err_3d_m'] + [f'err_{c}' for c in target_cols]].describe().T.round(3))

## 7. Первый baseline на траектории

Локальная метрика `dx/dy/dz` не показывает, как ошибка будет копиться при навигации без GPS. Поэтому ниже берем test-часть, выбираем неперекрывающиеся 5-секундные окна и накапливаем предсказанные Ridge-смещения от стартовой точки test-сегмента. Получившийся open-loop rollout накладываем на POS-траекторию.

In [ ]:
BASELINE_OUT = ROOT / 'artifacts' / 'generated' / 'baseline_trajectory'
BASELINE_OUT.mkdir(parents=True, exist_ok=True)

baseline_view = features.loc[
    test_mask,
    ['time_s', 'time_rel_s', 'east_m_ref', 'north_m_ref', 'up_m_ref'] + target_cols,
].copy().reset_index(drop=True)
for i, col in enumerate(target_cols):
    baseline_view[f'pred_{col}'] = pred_ridge[:, i]

baseline_view['true_east_end_m'] = baseline_view['east_m_ref'] + baseline_view['dx_east_m']
baseline_view['true_north_end_m'] = baseline_view['north_m_ref'] + baseline_view['dy_north_m']
baseline_view['true_up_end_m'] = baseline_view['up_m_ref'] + baseline_view['dz_up_m']
baseline_view['pred_east_end_m'] = baseline_view['east_m_ref'] + baseline_view['pred_dx_east_m']
baseline_view['pred_north_end_m'] = baseline_view['north_m_ref'] + baseline_view['pred_dy_north_m']
baseline_view['pred_up_end_m'] = baseline_view['up_m_ref'] + baseline_view['pred_dz_up_m']
baseline_view['local_error_3d_m'] = np.sqrt(
    (baseline_view['pred_east_end_m'] - baseline_view['true_east_end_m'])**2
    + (baseline_view['pred_north_end_m'] - baseline_view['true_north_end_m'])**2
    + (baseline_view['pred_up_end_m'] - baseline_view['true_up_end_m'])**2
)

selected_indices = []
next_time = -np.inf
for idx, row in baseline_view.iterrows():
    if row['time_rel_s'] >= next_time:
        selected_indices.append(idx)
        next_time = row['time_rel_s'] + horizon_s

roll_windows = baseline_view.iloc[selected_indices].copy().reset_index(drop=True)
pred_e = float(roll_windows.loc[0, 'east_m_ref'])
pred_n = float(roll_windows.loc[0, 'north_m_ref'])
pred_u = float(roll_windows.loc[0, 'up_m_ref'])
rollout_rows = []
for step, row in roll_windows.iterrows():
    start_e, start_n, start_u = pred_e, pred_n, pred_u
    pred_e += row['pred_dx_east_m']
    pred_n += row['pred_dy_north_m']
    pred_u += row['pred_dz_up_m']
    true_e = row['true_east_end_m']
    true_n = row['true_north_end_m']
    true_u = row['true_up_end_m']
    err_h = np.hypot(pred_e - true_e, pred_n - true_n)
    err_3d = np.sqrt((pred_e - true_e)**2 + (pred_n - true_n)**2 + (pred_u - true_u)**2)
    rollout_rows.append({
        'step': step + 1,
        'time_rel_s': row['time_rel_s'] + horizon_s,
        'start_pred_east_m': start_e,
        'start_pred_north_m': start_n,
        'start_pred_up_m': start_u,
        'pred_east_m': pred_e,
        'pred_north_m': pred_n,
        'pred_up_m': pred_u,
        'true_east_m': true_e,
        'true_north_m': true_n,
        'true_up_m': true_u,
        'horizontal_error_m': err_h,
        'error_3d_m': err_3d,
    })

rollout = pd.DataFrame(rollout_rows)
rollout_metrics = pd.DataFrame([{
    'steps': len(rollout),
    'duration_s': rollout['time_rel_s'].iloc[-1] - rollout['time_rel_s'].iloc[0],
    'final_horizontal_error_m': rollout['horizontal_error_m'].iloc[-1],
    'final_3d_error_m': rollout['error_3d_m'].iloc[-1],
    'mean_3d_error_m': rollout['error_3d_m'].mean(),
    'max_3d_error_m': rollout['error_3d_m'].max(),
}])

baseline_view.to_csv(BASELINE_OUT / 'ridge_local_windows.csv', index=False)
rollout.to_csv(BASELINE_OUT / 'ridge_sparse_rollout.csv', index=False)
rollout_metrics.to_csv(BASELINE_OUT / 'ridge_sparse_rollout_metrics.csv', index=False)

display(rollout_metrics.round(3))
print(f'Saved rollout: {(BASELINE_OUT / "ridge_sparse_rollout.csv").relative_to(ROOT)}')
print(f'Saved local windows: {(BASELINE_OUT / "ridge_local_windows.csv").relative_to(ROOT)}')

In [ ]:
fig = plt.figure(figsize=(17, 8), constrained_layout=True)
grid = fig.add_gridspec(2, 2, width_ratios=[1.35, 1.0])
ax_route = fig.add_subplot(grid[:, 0])
ax_height = fig.add_subplot(grid[0, 1])
ax_error = fig.add_subplot(grid[1, 1], sharex=ax_height)

rollout_start_time = rollout['time_rel_s'].iloc[0] - horizon_s
true_route_e = np.r_[rollout['start_pred_east_m'].iloc[0], rollout['true_east_m'].to_numpy()]
true_route_n = np.r_[rollout['start_pred_north_m'].iloc[0], rollout['true_north_m'].to_numpy()]
true_route_u = np.r_[rollout['start_pred_up_m'].iloc[0], rollout['true_up_m'].to_numpy()]
pred_route_e = np.r_[rollout['start_pred_east_m'].iloc[0], rollout['pred_east_m'].to_numpy()]
pred_route_n = np.r_[rollout['start_pred_north_m'].iloc[0], rollout['pred_north_m'].to_numpy()]
pred_route_u = np.r_[rollout['start_pred_up_m'].iloc[0], rollout['pred_up_m'].to_numpy()]
rollout_time = np.r_[rollout_start_time, rollout['time_rel_s'].to_numpy()]

ax_route.plot(pos['east_m'], pos['north_m'], color='0.82', linewidth=1.5, label='full POS route')
ax_route.plot(true_route_e, true_route_n, color='tab:blue', linewidth=2.4, label='POS test segment')
ax_route.plot(pred_route_e, pred_route_n, color='tab:red', linewidth=2.4, label='Ridge sparse rollout')
ax_route.scatter(true_route_e[0], true_route_n[0], color='black', s=45, label='shared rollout start')
ax_route.scatter(pred_route_e[-1], pred_route_n[-1], color='tab:red', s=55, marker='x', label='pred final')
ax_route.scatter(true_route_e[-1], true_route_n[-1], color='tab:blue', s=55, marker='x', label='true final')
ax_route.set_title('First Ridge baseline accumulated on trajectory')
ax_route.set_xlabel('east, m')
ax_route.set_ylabel('north, m')
ax_route.set_aspect('equal', adjustable='box')
ax_route.legend(fontsize=8)

ax_height.plot(rollout_time, true_route_u, color='tab:blue', label='POS up')
ax_height.plot(rollout_time, pred_route_u, color='tab:red', label='Ridge rollout up')
ax_height.set_title('Rollout height')
ax_height.set_ylabel('up, m')
ax_height.legend(fontsize=8)

ax_error.plot(rollout['time_rel_s'], rollout['horizontal_error_m'], color='tab:purple', label='horizontal error')
ax_error.plot(rollout['time_rel_s'], rollout['error_3d_m'], color='tab:orange', label='3D error')
ax_error.set_title('Accumulated rollout error')
ax_error.set_xlabel('time, s')
ax_error.set_ylabel('error, m')
ax_error.legend(fontsize=8)

trajectory_png = BASELINE_OUT / 'ridge_sparse_rollout_overlay.png'
fig.savefig(trajectory_png, dpi=160)
print(f'Saved image: {trajectory_png.relative_to(ROOT)}')
plt.show()

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
ax.plot(pos['east_m'], pos['north_m'], color='0.88', linewidth=1.2, label='full POS route')
ax.scatter(baseline_view['true_east_end_m'], baseline_view['true_north_end_m'], s=8, alpha=0.25, label='true 5s endpoints')
ax.scatter(baseline_view['pred_east_end_m'], baseline_view['pred_north_end_m'], s=8, alpha=0.25, label='predicted 5s endpoints')
ax.set_title('Local 5-second endpoint predictions')
ax.set_xlabel('east, m')
ax.set_ylabel('north, m')
ax.set_aspect('equal', adjustable='box')
ax.legend(fontsize=8)
local_png = BASELINE_OUT / 'ridge_local_endpoint_overlay.png'
fig.savefig(local_png, dpi=160)
print(f'Saved image: {local_png.relative_to(ROOT)}')
plt.show()

## 8. Подготовка к проверке гипотез

Дальше имеет смысл проверять гипотезы в таком порядке:

1. **Target/horizon**: сравнить горизонты `1s`, `3s`, `5s` и отдельно горизонтальную цель `dx/dy` без `dz`.
2. **Фильтр движения**: исключить почти неподвижные окна, например где `sqrt(dx^2 + dy^2) < 1 m` на горизонте.
3. **Признаки**: добавить моторы `RCOU_motor_features`, батарею `BAT`, а также body-to-ENU ускорения с компенсацией гравитации.
4. **Валидация**: заменить один split `80/20` на rolling validation по временным folds.
5. **Rollout**: оценивать не только локальную ошибку `dx/dy/dz`, но и накопленный drift траектории.

В репозитории уже есть более зрелые скрипты и отчеты для этих пунктов: `src/run_dataflash_sequence_baseline.py`, `reports/final_dataflash_report.md`, `reports/experiments/dataflash_rollout_summary.md`. Этот ноутбук нужен как прозрачная стартовая площадка для понимания данных и быстрой проверки следующих идей.